# Centauri Bloom — Unified trainer
Colab or Kaggle. On Kaggle import this notebook, enable Internet and a GPU accelerator, attach the private contributor dataset for Jaron, and enable the HF_TOKEN secret. Edit settings in cell 1. Outputs are saved in /kaggle/working; save a notebook version or download completed artifacts before ending the session. On Colab outputs go to Drive. Candidate weights are submitted as a Hugging Face PR; review before merging. Never publish raw contributor data or secrets.


In [ ]:
#@title 1. Choose training settings
DATASET = "Jaron" #@param ["Jaron","FineWeb-Edu + TinyStories","FineWeb-Edu","TinyStories","Cosmopedia","Public-domain literature","Public-domain + TinyStories","Arithmetic instruction pilot"]
STEPS = 3000 #@param [100,500,1000,1500,3000,5000,10000] {type:"raw"}
#@markdown Review the selected dataset's terms and confirm you may train on it and submit candidate weights.
CONFIRM_DATA_PERMISSION = False #@param {type:"boolean"}
#@markdown Optional: recover processed windows from a previous unified run before its first checkpoint.
RECOVER_RUN_DIR = "" #@param {type:"string"}
JARON_FILE = "" #@param {type:"string"}
# Kaggle: optional exact path to attached private JSON; blank auto-detects.
if not CONFIRM_DATA_PERMISSION:
    raise ValueError("Confirm dataset permission before running.")
print("Selected:",DATASET,"|",STEPS,"optimizer step attempts")


In [ ]:
#@title 2. Set up GPU and Google Drive
import os,sys,subprocess,json
from pathlib import Path
from datetime import datetime,timezone
import shutil
IN_KAGGLE=Path('/kaggle/working').is_dir()
if not IN_KAGGLE:
    from google.colab import drive
os.environ['HF_HUB_DISABLE_XET']='1'
def checked(args):
    p=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    print(p.stdout,flush=True)
    if p.returncode:
        raise RuntimeError(f'Setup failed (exit {p.returncode}). See full diagnostic above.')
# Detect GPU before changing any dependencies.
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU detected. Enable GPU in Kaggle Settings or Colab runtime settings.')
checked(['nvidia-smi'])
REPO=Path('/kaggle/working/centauri-bloom' if IN_KAGGLE else '/content/centauri-bloom')
if not (REPO/'.git').exists():
    checked(['git','clone','--depth','1','https://github.com/InterstellarNerd8760/centauri-bloom.git',str(REPO)])
else:
    checked(['git','-C',str(REPO),'pull','--ff-only','origin','main'])
os.chdir(REPO)
# Preserve Colab's CUDA-enabled PyTorch and its matching dependencies.
checked([sys.executable,'-m','pip','install','--no-deps','-e','.'])
checked([sys.executable,'-m','pip','install','tokenizers','datasets','huggingface_hub','ipywidgets','safetensors'])
checked([sys.executable,'-c','from bloom.notebook_workflow import preflight; preflight("cuda")'])
from bloom.notebook_workflow import submit_candidate,sha256
from bloom.trainer import REPO_ID
from bloom.trainer_data import JARON_SHA,RECIPES
if DATASET not in RECIPES or type(STEPS) is not int or not 1<=STEPS<=10000:
    raise ValueError('Invalid dataset or step budget.')
if IN_KAGGLE:
    BASE=Path('/kaggle/working/centauri-bloom-runs')
else:
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/centauri-bloom-runs')
BASE.mkdir(parents=True,exist_ok=True)
slug=DATASET.lower().replace(' + ','-').replace(' ','-')
OUT_DIR=Path(RECOVER_RUN_DIR) if RECOVER_RUN_DIR else BASE/('bloom-'+slug+'-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
HF_REPO_ID=REPO_ID
print('Output:',OUT_DIR)


In [ ]:
#@title 3. Authorize Hugging Face
from getpass import getpass
from bloom.notebook_workflow import auth_client
try:
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
except Exception:
    token=None
if not token:
    token=getpass('Hugging Face write token: ')
AUTH_API=auth_client(HF_REPO_ID,token)
del token
print('Authenticated. Candidate PR only.')


In [ ]:
#@title 4. Load latest model, prepare selected data, train and evaluate
if AUTH_API is None:
    raise RuntimeError('Tap Authorize upload in the preceding cell.')
jaron_path=None
if DATASET=='Jaron' and not RECOVER_RUN_DIR:
    if JARON_FILE:
        jaron_path=Path(JARON_FILE)
    elif IN_KAGGLE:
        matches=list(Path('/kaggle/input').rglob('bloom_instruction_expanded.json'))
        matches=[p for p in matches if sha256(p)==JARON_SHA]
        if len(matches)!=1:
            raise ValueError('Attach the original private contributor dataset or set JARON_FILE to its exact path.')
        jaron_path=matches[0]
    else:
        folder=Path('/content/drive/MyDrive/centauri-bloom-data')
        folder.mkdir(parents=True,exist_ok=True)
        jaron_path=folder/'bloom_instruction_expanded.json'
        if not jaron_path.exists():
            from google.colab import files
            uploaded=files.upload()
            raw=uploaded.get(jaron_path.name)
            if raw is None:
                raise ValueError('Select the original contributor JSON.')
            import hashlib
            if hashlib.sha256(raw).hexdigest()!=JARON_SHA:
                raise ValueError('Contributor file hash mismatch.')
            jaron_path.write_bytes(raw)
    if sha256(jaron_path)!=JARON_SHA:
        raise ValueError('Contributor file hash mismatch.')
# Train in a fresh process, avoiding stale notebook imports and Dynamo state.
env=os.environ.copy()
env['HF_TOKEN']=AUTH_API.token
payload=json.dumps(dict(recipe=DATASET,steps=STEPS,repo_dir=str(REPO),out_dir=str(OUT_DIR),jaron_path=str(jaron_path) if jaron_path else None,recover=bool(RECOVER_RUN_DIR)))
script="import json,sys,os; from huggingface_hub import HfApi; from bloom.notebook_workflow import preflight; from bloom.trainer import run_training; preflight('cuda'); run_training(HfApi(token=os.environ['HF_TOKEN']),**json.loads(sys.argv[1]))"
try:
    result=subprocess.run([sys.executable,'-u','-c',script,payload],env=env)
finally:
    env.pop('HF_TOKEN',None)
if result.returncode:
    raise RuntimeError('Training failed. See diagnostic above; saved files remain in the output directory.')
metrics=json.loads((OUT_DIR/'metrics.json').read_text())


In [ ]:
#@title 5. Automatically submit the completed candidate
if AUTH_API is None:
    raise RuntimeError('Authorize upload first.')
metrics=json.loads((OUT_DIR/'metrics.json').read_text())
if metrics['additional_steps']!=STEPS or metrics['dataset']!=DATASET:
    raise RuntimeError('Run settings differ from completed candidate.')
submit_candidate(AUTH_API,HF_REPO_ID,OUT_DIR,STEPS,metrics['parent_revision'],metrics['run_name'])
print('Review and merge the candidate in Hugging Face before the next run.')
